# 00 — Generación del dataset sintético
**Proyecto:** LogiSmart — Predicción de retrasos en entregas de última milla  
**Tarea WBS:** previa a 1.2.1 (Extracción y exploración) — insumo base para toda la Fase 2 y 3  
**Responsable:** Data Engineer

Como el caso no entrega un dataset real, se genera un dataset sintético que respeta:
- El volumen y las columnas descritas en el caso (versión reducida a 8.000 filas para fines del prototipo; el caso real son 1.800.000).
- La tasa de retraso global reportada (~15%).
- El sesgo de muestreo identificado en EP1: comunas periféricas con muchos menos registros históricos.
- Un problema real de calidad de datos (15% de nulos en `comuna_origen`) que se resolverá en la Fase 3 (tarea 1.3.1).

In [2]:
"""
00_generacion_datos — LogiSmart
Genera un dataset sintético de entregas de última milla, consistente con
el Caso 3 (SCY1102) y con los sesgos ya identificados en EP1/EP3:
  - Comunas periféricas con muy pocos registros históricos (sesgo de muestreo)
  - Conductores que concentran rutas en ciertas comunas
  - ~15% de entregas con retraso (desbalance de clase real del caso)
  - Nulos en comuna_origen (problema de calidad de datos a resolver en EDA/Fase 3)
No incluye pseudonimización todavía: eso corresponde a la tarea 1.3.2
(se aplica en el notebook/paso siguiente, sobre este dataset crudo).
"""

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os

rng = np.random.default_rng(42)

N = 180000  # tamaño manejable para prototipo (el caso real son 1.8M)

# --- Comunas: Metropolitana + Valparaíso, con desbalance deliberado ---
# "núcleo" = alto volumen histórico, "periféricas" = bajo volumen (sesgo de muestreo de EP1)
comunas_nucleo = ["Santiago", "Providencia", "Ñuñoa", "Las Condes", "Maipú",
                   "San Miguel", "Viña del Mar", "Valparaíso"]
comunas_perifericas = ["Til Til", "Melipilla", "Quilpué", "Casablanca", "San Pedro"]

# pesos: núcleo concentra ~92% de los registros, periféricas solo ~8%
pesos_comuna = (
    [0.92 / len(comunas_nucleo)] * len(comunas_nucleo) +
    [0.08 / len(comunas_perifericas)] * len(comunas_perifericas)
)
todas_comunas = comunas_nucleo + comunas_perifericas

comuna_origen = rng.choice(todas_comunas, size=N, p=pesos_comuna)
comuna_destino = rng.choice(todas_comunas, size=N, p=pesos_comuna)

# --- Conductores: 150 conductores, concentración de rutas (no uniforme) ---
n_conductores = 150
# algunos conductores concentran muchas más rutas que otros (ley de potencia simplificada)
pesos_conductor = rng.exponential(scale=1.0, size=n_conductores)
pesos_conductor = pesos_conductor / pesos_conductor.sum()
id_conductor = rng.choice([f"COND-{i:04d}" for i in range(n_conductores)], size=N, p=pesos_conductor)

experiencia_conductor_anios = rng.integers(0, 15, size=N)

# --- Fecha / hora programada: últimos 6 meses ---
fecha_fin = datetime(2026, 10, 31)
fecha_inicio = fecha_fin - timedelta(days=182)
dias_random = rng.integers(0, 182, size=N)
fecha = [fecha_inicio + timedelta(days=int(d)) for d in dias_random]
hora_programada = [f"{h:02d}:{m:02d}" for h, m in zip(rng.integers(8, 19, N), rng.integers(0, 60, N))]

# --- Variables operacionales ---
tipo_vehiculo = rng.choice(["Furgón", "Camioneta", "Moto"], size=N, p=[0.5, 0.35, 0.15])
distancia_estimada_km = np.round(rng.gamma(shape=2.0, scale=6.0, size=N) + 1, 1)
cantidad_paquetes = rng.integers(1, 40, size=N)
peso_kg = np.round(cantidad_paquetes * rng.uniform(0.5, 3.5, size=N), 1)
volumen_m3 = np.round(peso_kg * rng.uniform(0.01, 0.03, size=N), 3)
cantidad_paradas = rng.integers(1, 12, size=N)

duracion_estimada_min = np.round(distancia_estimada_km * rng.uniform(2.0, 3.5, size=N) + cantidad_paradas * 2, 0)

# --- Generación del RETRASO (target) ---
# Probabilidad base ~12%, se incrementa en comunas periféricas (menos datos = rutas
# menos optimizadas/conocidas) y con conductores de baja experiencia.
es_periferica = np.isin(comuna_origen, comunas_perifericas) | np.isin(comuna_destino, comunas_perifericas)
prob_retraso = 0.10 + 0.22 * es_periferica + 0.05 * (experiencia_conductor_anios < 2) \
               + 0.04 * (cantidad_paradas > 8)
prob_retraso = np.clip(prob_retraso, 0.03, 0.85)
retraso = rng.binomial(1, prob_retraso)

# Duración real: si hay retraso, se extiende sobre la estimada
factor_extra = np.where(retraso == 1, rng.uniform(1.25, 1.8, size=N), rng.uniform(0.85, 1.15, size=N))
duracion_real_min = np.round(duracion_estimada_min * factor_extra, 0)

# hora_salida_real / hora_entrega_real (simplificado como offset en minutos desde hora programada)
hora_salida_real = [f"{(int(h)+rng.integers(0,2))%24:02d}:{(int(m)+rng.integers(0,30))%60:02d}"
                     for h, m in zip([int(h.split(':')[0]) for h in hora_programada],
                                      [int(h.split(':')[1]) for h in hora_programada])]

incidencias = np.where(retraso == 1,
                        rng.choice(["Tráfico", "Dirección incorrecta", "Vehículo no disponible",
                                    "Cliente ausente", "Clima"], size=N),
                        "Ninguna")

estado_final = np.where(retraso == 1, "Retrasada", "A tiempo")

df = pd.DataFrame({
    "id_entrega": [f"ENT-{i:06d}" for i in range(N)],
    "fecha": fecha,
    "hora_programada": hora_programada,
    "comuna_origen": comuna_origen,
    "comuna_destino": comuna_destino,
    "distancia_estimada_km": distancia_estimada_km,
    "tipo_vehiculo": tipo_vehiculo,
    "id_conductor": id_conductor,
    "experiencia_conductor_anios": experiencia_conductor_anios,
    "cantidad_paquetes": cantidad_paquetes,
    "peso_kg": peso_kg,
    "volumen_m3": volumen_m3,
    "duracion_estimada_min": duracion_estimada_min,
    "duracion_real_min": duracion_real_min,
    "cantidad_paradas": cantidad_paradas,
    "hora_salida_real": hora_salida_real,
    "incidencias": incidencias,
    "estado_final": estado_final,
    "retraso": retraso,  # target: 1 = retrasada, 0 = a tiempo
})

# --- Inyección deliberada de nulos en comuna_origen (~15%) ---
# Simula el problema de calidad de datos que la Fase 3 (1.3.1) debe resolver.
idx_nulos = rng.choice(df.index, size=int(0.15 * N), replace=False)
df.loc[idx_nulos, "comuna_origen"] = np.nan

os.makedirs("data", exist_ok=True)

df.to_csv("data/logismart_entregas_sinteticas.csv", index=False, encoding='utf-8')

print(f"Dataset generado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Tasa de retraso global: {df['retraso'].mean():.1%}")
print(f"Nulos en comuna_origen: {df['comuna_origen'].isna().sum()} ({df['comuna_origen'].isna().mean():.1%})")
print(f"Registros en comunas periféricas: {es_periferica.sum()} ({es_periferica.mean():.1%})")
print(f"Tasa de retraso en comunas periféricas: {df.loc[es_periferica, 'retraso'].mean():.1%}")
print(f"Tasa de retraso en comunas núcleo: {df.loc[~es_periferica, 'retraso'].mean():.1%}")


Dataset generado: 180000 filas, 19 columnas
Tasa de retraso global: 15.1%
Nulos en comuna_origen: 27000 (15.0%)
Registros en comunas periféricas: 27690 (15.4%)
Tasa de retraso en comunas periféricas: 33.6%
Tasa de retraso en comunas núcleo: 11.7%


## Notas para el siguiente paso (EDA — tarea 1.2.1 / 1.2.2 / 1.2.3)

- El archivo generado queda en `data/logismart_entregas_sinteticas.csv`.
- **No se aplicó pseudonimización todavía**: `id_conductor` está en texto plano a propósito. El hash salado corresponde a la tarea **1.3.2**, en la fase de preparación, no aquí.
- Los nulos de `comuna_origen` quedaron intencionalmente sin tratar: documentar su tratamiento es parte de la tarea **1.3.1**.
- La diferencia de tasa de retraso entre comunas núcleo y periféricas ya está impresa arriba: úsenla como punto de partida del diagnóstico de sesgo (tarea **1.2.3**).